<a href="https://colab.research.google.com/github/Simran2622/text-summarization-using-neural-network/blob/main/text_summarization_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install transformers
!pip install datasets
!pip install rouge-score
!pip install sentencepiece

In [ ]:
from datasets import load_dataset

# Load CNN/DailyMail dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)

In [ ]:
# Create smaller subsets for faster training

train_data = dataset["train"].select(range(2000))
validation_data = dataset["validation"].select(range(500))
test_data = dataset["test"].select(range(500))

print("Train samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Test samples:", len(test_data))

In [ ]:
# Inspect one example

example = train_data[0]

print("ARTICLE:\n")
print(example["article"])

print("\n\nSUMMARY:\n")
print(example["highlights"])

In [ ]:
# Generate summary using BART directly

text = train_data[0]["article"]

inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=130,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("ARTICLE:\n")
print(text)

print("\n\nGENERATED SUMMARY:\n")
print(summary)

In [ ]:
from rouge_score import rouge_scorer

# Get reference summary
reference = train_data[0]["highlights"]

# Get generated summary
generated = summary

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

scores = scorer.score(reference, generated)

print("REFERENCE SUMMARY:\n")
print(reference)

print("\n\nGENERATED SUMMARY:\n")
print(generated)

print("\n\nROUGE SCORES:\n")
print(scores)

In [ ]:
from tqdm import tqdm

rouge1 = []
rouge2 = []
rougeL = []

for i in tqdm(range(50)):  # evaluate 50 test samples

    article = test_data[i]["article"]
    reference = test_data[i]["highlights"]

    inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=130,
        min_length=30,
        num_beams=4
    )

    generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    score = scorer.score(reference, generated)

    rouge1.append(score["rouge1"].fmeasure)
    rouge2.append(score["rouge2"].fmeasure)
    rougeL.append(score["rougeL"].fmeasure)

print("Average ROUGE-1:", sum(rouge1)/len(rouge1))
print("Average ROUGE-2:", sum(rouge2)/len(rouge2))
print("Average ROUGE-L:", sum(rougeL)/len(rougeL))

In [ ]:
import matplotlib.pyplot as plt

# Your results
rouge1_score = sum(rouge1)/len(rouge1)
rouge2_score = sum(rouge2)/len(rouge2)
rougeL_score = sum(rougeL)/len(rougeL)

labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
scores = [rouge1_score, rouge2_score, rougeL_score]

plt.figure(figsize=(6,4))
plt.bar(labels, scores)

plt.title("ROUGE Score Evaluation of Neural Text Summarization Model")
plt.ylabel("Score")
plt.xlabel("Evaluation Metrics")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(labels, scores)

plt.title("ROUGE Score Evaluation of Neural Text Summarization Model")
plt.ylabel("Score")
plt.xlabel("Evaluation Metrics")

plt.savefig("rouge_scores_graph.png", dpi=300)

plt.show()

In [ ]:
import graphviz

diagram = graphviz.Digraph()

diagram.node('A', 'Input Article')
diagram.node('B', 'Tokenizer')
diagram.node('C', 'BART Neural Network')
diagram.node('D', 'Generated Summary')
diagram.node('E', 'ROUGE Evaluation')

diagram.edges(['AB', 'BC', 'CD', 'DE'])

diagram

In [ ]:
import matplotlib.pyplot as plt
from datasets import load_dataset # Added import

# Load CNN/DailyMail dataset and create smaller subsets (duplicated from previous cells to make this cell runnable)
dataset = load_dataset("cnn_dailymail", "3.0.0")
train_data = dataset["train"].select(range(2000))
validation_data = dataset["validation"].select(range(500))
test_data = dataset["test"].select(range(500))

# Dataset sizes
train_size = len(train_data)
validation_size = len(validation_data)
test_size = len(test_data)

labels = ["Train", "Validation", "Test"]
sizes = [train_size, validation_size, test_size]

plt.figure(figsize=(6,4))
plt.bar(labels, sizes)

plt.title("Dataset Distribution for Text Summarization Experiment")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Samples")

plt.savefig("dataset_distribution.png", dpi=300)

plt.show()